In [1]:
# ============================================================
# BEDLOCATION (ICU-focused): Generate BEDLOCATION.csv for MIMIC-IV
# ============================================================

import os
import pandas as pd

# Directories
input_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/builtdata/csv_exports"
output_dir = "/hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files"
os.makedirs(output_dir, exist_ok=True)

# Load source tables
icustays_path = os.path.join(input_dir, "icu_icustays.csv")
transfers_path = os.path.join(input_dir, "hosp_transfers.csv")

icustays = pd.read_csv(icustays_path)
transfers = pd.read_csv(transfers_path)

print("✅ icu_icustays.csv loaded:", icustays.shape)
print("✅ hosp_transfers.csv loaded:", transfers.shape)


✅ icu_icustays.csv loaded: (94458, 8)
✅ hosp_transfers.csv loaded: (2413581, 7)


In [2]:

# ----------------------------------------------------------------
# Join icustays with transfers to get bed_unit info
# ----------------------------------------------------------------
# We'll take the first matching careunit per (subject_id, hadm_id)
careunit_lookup = (
    transfers
    .dropna(subset=["careunit"])  # only rows with careunit
    .groupby(["subject_id", "hadm_id"])["careunit"]
    .first()
    .reset_index()
)

print("✅ careunit lookup table created:", careunit_lookup.shape)


✅ careunit lookup table created: (546025, 3)


In [3]:

# Merge bed_unit from transfers into icustays
bedlocation = (
    icustays
    .merge(careunit_lookup, how="left", on=["subject_id", "hadm_id"])
    .rename(columns={
        "subject_id": "pat_id",
        "hadm_id": "csn",
        "intime": "bed_location_start",
        "outtime": "bed_location_end",
        "careunit": "bed_unit"
    })
)

print("✅ BEDLOCATION merged:", bedlocation.shape)

# Clean columns
bedlocation["bed_unit"] = bedlocation["bed_unit"].fillna("Not Recorded")

# Convert times
bedlocation["bed_location_start"] = pd.to_datetime(bedlocation["bed_location_start"], errors="coerce")
bedlocation["bed_location_end"] = pd.to_datetime(bedlocation["bed_location_end"], errors="coerce")

# ✅ Only keep required columns
bedlocation = bedlocation.loc[:, ["csn", "pat_id", "bed_unit", "bed_location_start", "bed_location_end"]]

# Final checks
print("✅ After cleaning:", bedlocation.shape)
print("Missing values:\n", bedlocation.isna().sum())
print("Sample rows:\n", bedlocation.head())

# Save
out_path = os.path.join(output_dir, "BEDLOCATION.csv")
bedlocation.to_csv(out_path, index=False)

print(f"✅ BEDLOCATION.csv saved to {out_path} with shape {bedlocation.shape}")
print("Final columns:", bedlocation.columns.tolist())

✅ BEDLOCATION merged: (94458, 9)
✅ After cleaning: (94458, 5)
Missing values:
 csn                    0
pat_id                 0
bed_unit               0
bed_location_start     0
bed_location_end      14
dtype: int64
Sample rows:
         csn    pat_id              bed_unit  bed_location_start  \
0  29079034  10000032  Emergency Department 2180-07-23 14:00:00   
1  25860671  10000690              Medicine 2150-11-02 19:37:00   
2  26913865  10000980  Emergency Department 2189-06-27 08:42:00   
3  24597018  10001217             Neurology 2157-11-20 19:18:02   
4  27703517  10001217             Neurology 2157-12-19 15:42:24   

     bed_location_end  
0 2180-07-23 23:50:47  
1 2150-11-06 17:03:17  
2 2189-06-27 20:38:27  
3 2157-11-21 22:08:00  
4 2157-12-20 14:27:41  
✅ BEDLOCATION.csv saved to /hpc/home/yy450/link_kamaleswaranlab/mimic_iv/mimic_flat_files/BEDLOCATION.csv with shape (94458, 5)
Final columns: ['csn', 'pat_id', 'bed_unit', 'bed_location_start', 'bed_location_end']
